Is DEI / AI discourse in Information Technology firms distinctive, once we control for general market-wide language inflation?

In [ ]:
# What this cell does:
# End-to-end DEI + AI token-weighted disclosure intensity (hits per 1,000 tokens)
# -> Load AI_DEI_Lexicon.yml and extract AI + DEI terms
# -> Load filings, derive year, normalise text to STRING (handles list[str] and str)
# -> Collapse sections -> firm-year document
# -> Compute capped-IDF weights across firm-year docs
# -> Score token-weighted hits, normalise by doc_length
# -> Aggregate by year: All sectors vs IT
# -> Plot + save PNGs reproducibly

from __future__ import annotations

from pathlib import Path
import os
import re
import math
import yaml

import polars as pl
import matplotlib.pyplot as plt


# --------------------
# Config
# --------------------
PARQUET_FILE = Path("./spy_10k_2015_present.parquet")
LEXICON_FILE = Path("./AI_DEI_Lexicon.yml")

YEAR_MIN, YEAR_MAX = 2015, 2025
IT_SECTOR_NAME = "Information Technology"

TEXT_COL = "text"
SECTOR_COL = "gics_sector"
DATE_COL = "filing_date"
PERIOD_COL = "filing_period"
FIRM_COL = "ticker"

IDF_CAP = 8.0

OUT_DIR = Path("./outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_DEI = OUT_DIR / "it_vs_all_DEI_token_weighted_04.png"
OUT_AI  = OUT_DIR / "it_vs_all_AI_token_weighted_04.png"


# --------------------
# Helpers
# --------------------
def _clean_term(t: str) -> str:
    t = t.strip()
    t = re.sub(r"\s+", " ", t)
    t = t.strip(" ,;:.")
    return t

def _extract_strings(node):
    out = []
    if node is None:
        return out
    if isinstance(node, str):
        out.append(node)
    elif isinstance(node, list):
        for x in node:
            out.extend(_extract_strings(x))
    elif isinstance(node, dict):
        for v in node.values():
            out.extend(_extract_strings(v))
    return out

def _find_terms_any(obj, want: str) -> list[str]:
    subkeys = {"terms", "phrases", "patterns", "lexicon", "keywords"}

    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).lower() == want.lower():
                terms = [_clean_term(x) for x in _extract_strings(v) if _clean_term(x)]
                if terms:
                    return sorted(set(terms))
                if isinstance(v, dict):
                    for sk in subkeys:
                        if sk in v:
                            terms = [_clean_term(x) for x in _extract_strings(v[sk]) if _clean_term(x)]
                            if terms:
                                return sorted(set(terms))
        for v in obj.values():
            found = _find_terms_any(v, want)
            if found:
                return found

    if isinstance(obj, list):
        for x in obj:
            found = _find_terms_any(x, want)
            if found:
                return found

    return []

def _regex_for_term(t: str) -> str:
    s = re.escape(_clean_term(t)).replace(r"\ ", r"\s+")
    return r"(?i)(^|[^A-Za-z0-9])(" + s + r")([^A-Za-z0-9]|$)"

def _idf(df_term: int, N: int, cap: float) -> float:
    val = math.log((N + 1) / (df_term + 1)) + 1.0
    return min(val, cap)

def _idf_weights(df: pl.DataFrame, terms: list[str], cap: float) -> dict[str, float]:
    N_docs = df.height
    weights = {}
    for t in terms:
        rt = _regex_for_term(t)
        df_term = df.select(pl.col(TEXT_COL).str.contains(rt).sum().alias("df")).item()
        weights[t] = _idf(int(df_term), int(N_docs), cap)
    return weights

def _weighted_hits_expr(terms: list[str], weights: dict[str, float]) -> pl.Expr:
    exprs = []
    for t in terms:
        rt = _regex_for_term(t)
        exprs.append(pl.col(TEXT_COL).str.count_matches(rt) * pl.lit(float(weights[t])))
    if not exprs:
        return pl.lit(0.0)
    out = exprs[0]
    for e in exprs[1:]:
        out = out + e
    return out

def _save_lineplot(years, y_all, y_it, title, ylabel, out_path: Path):
    if out_path.exists():
        os.remove(out_path)
    plt.figure(figsize=(12, 8))
    plt.plot(years, y_all, label="All sectors")
    if any(v is not None for v in y_it):
        plt.plot(years, y_it, label="Information Technology")
    plt.title(title)
    plt.xlabel("Year")
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


# --------------------
# Lexicon
# --------------------
if not LEXICON_FILE.exists():
    raise FileNotFoundError("AI_DEI_Lexicon.yml not found in the working directory.")

lex = yaml.safe_load(LEXICON_FILE.read_text(encoding="utf-8"))

AI_TERMS  = _find_terms_any(lex, "AI")
DEI_TERMS = _find_terms_any(lex, "DEI")

print("AI terms:", len(AI_TERMS), "DEI terms:", len(DEI_TERMS))
print("AI sample:", AI_TERMS[:20])
print("DEI sample:", DEI_TERMS[:20])

if not AI_TERMS or not DEI_TERMS:
    raise ValueError("Lexicon extraction failed — no terms found.")


# --------------------
# Load + year
# --------------------
df = pl.read_parquet(PARQUET_FILE)

df = df.with_columns(
    pl.coalesce([
        pl.col(DATE_COL).cast(pl.Date, strict=False).dt.year(),
        pl.col(PERIOD_COL).cast(pl.Utf8).str.extract(r"(\d{4})", 1).cast(pl.Int64),
    ]).alias("year")
).filter(
    pl.col("year").is_between(YEAR_MIN, YEAR_MAX)
)

# --------------------
# NORMALISE TEXT TO STRING (handles list[str] vs str)
# --------------------
# What this does: Polars Expr has no .dtype, so branch on the column dtype from df.schema
df = df.with_columns(
text_dtype = df.schema.get(TEXT_COL)

if isinstance(text_dtype, pl.List):
    df = df.with_columns(
        pl.col(TEXT_COL).list.join(" ").alias(TEXT_COL)
    )
else:
    df = df.with_columns(
        pl.col(TEXT_COL).cast(pl.Utf8, strict=False).alias(TEXT_COL)
    )

    .then(pl.col(TEXT_COL).list.join(" "))
    .otherwise(pl.col(TEXT_COL).cast(pl.Utf8))
    .alias(TEXT_COL)
)

# Collapse sections -> firm-year doc
df_fy = (
    df.group_by([FIRM_COL, "year", SECTOR_COL])
      .agg(pl.concat_str(pl.col(TEXT_COL), separator="\n").alias(TEXT_COL))
)

# Compute doc_length
df_fy = df_fy.with_columns(
    pl.col(TEXT_COL).str.split(by=r"\s+").list.len().cast(pl.Int64).alias("doc_length")
).filter(pl.col("doc_length") > 0)

it_mask = pl.col(SECTOR_COL).cast(pl.Utf8).str.contains(IT_SECTOR_NAME, literal=True)

print("Firm-years (all):", df_fy.height)
print("Firm-years (IT):", df_fy.filter(it_mask).height)


# --------------------
# Score
# --------------------
w_dei = _idf_weights(df_fy, DEI_TERMS, IDF_CAP)
w_ai  = _idf_weights(df_fy, AI_TERMS,  IDF_CAP)

scored = df_fy.with_columns(
    _weighted_hits_expr(DEI_TERMS, w_dei).alias("dei_hits_w"),
    _weighted_hits_expr(AI_TERMS,  w_ai ).alias("ai_hits_w"),
).with_columns(
    (pl.col("dei_hits_w") / pl.col("doc_length") * 1000.0).alias("dei_rate"),
    (pl.col("ai_hits_w")  / pl.col("doc_length") * 1000.0).alias("ai_rate"),
)

print(
    scored.select(
        pl.sum("dei_hits_w").alias("sum_dei_hits_w"),
        pl.sum("ai_hits_w").alias("sum_ai_hits_w"),
        pl.mean("dei_rate").alias("mean_dei_rate"),
        pl.mean("ai_rate").alias("mean_ai_rate"),
    )
)

# Aggregate by year
year_all = (
    scored.group_by("year")
    .agg(
        pl.mean("dei_rate").alias("dei_rate_all"),
        pl.mean("ai_rate").alias("ai_rate_all"),
    )
    .sort("year")
)

year_it = (
    scored.filter(it_mask)
    .group_by("year")
    .agg(
        pl.mean("dei_rate").alias("dei_rate_it"),
        pl.mean("ai_rate").alias("ai_rate_it"),
    )
    .sort("year")
)

ts = year_all.join(year_it, on="year", how="left").sort("year")

years = ts["year"].to_list()

_save_lineplot(
    years,
    ts["dei_rate_all"].to_list(),
    ts["dei_rate_it"].to_list(),
    "DEI disclosure intensity (token-weighted): IT vs All sectors",
    "DEI_rate (token-weighted; hits per 1,000 tokens)",
    OUT_DEI,
)

_save_lineplot(
    years,
    ts["ai_rate_all"].to_list(),
    ts["ai_rate_it"].to_list(),
    "AI disclosure intensity (token-weighted): IT vs All sectors",
    "AI_rate (token-weighted; hits per 1,000 tokens)",
    OUT_AI,
)

OUT_DEI, OUT_AI


AI terms: 95 DEI terms: 99
AI sample: ['(?i)\\ba\\.i\\.\\b', '(?i)\\baccountability\\b', '(?i)\\bagents?\\b', '(?i)\\bai[-\\s]+ethics\\b', '(?i)\\bai[-\\s]+fairness\\b', '(?i)\\bai[-\\s]+risk\\b', '(?i)\\bai[-\\s]+safety\\b', '(?i)\\bai\\b', '(?i)\\balgorithmic[-\\s]+bias\\b', '(?i)\\balgorithmic\\b', '(?i)\\balgorithms?\\b', '(?i)\\banomaly[-\\s]+detection\\b', '(?i)\\bartificial[-\\s]+intelligence\\b', '(?i)\\bautomation\\b', '(?i)\\bautonomous[-\\s]+agents?\\b', '(?i)\\bautonomous[-\\s]+systems?\\b', '(?i)\\bbias\\b', '(?i)\\bchatgpt\\b', '(?i)\\bclassification\\b', '(?i)\\bclustering\\b']
DEI sample: ['(?i)\\b(equal|fair)[-\\s]+treatment\\b', '(?i)\\b(equal|gender)[-\\s]+pay\\b', '(?i)\\b(first[-\\s]+nations)\\b', '(?i)\\b(inclusive|inclusivity)\\b', '(?i)\\b(unconscious|implicit)[-\\s]+bias\\b', '(?i)\\baccessibility\\b', '(?i)\\baccommodations?\\b', '(?i)\\baffirmative[-\\s]+action\\b', '(?i)\\bafrican[-\\s]+american\\b', '(?i)\\bage[-\\s]+discrimination\\b', '(?i)\\bageism\\b', 

AttributeError: 'Expr' object has no attribute 'dtype'